# Exercise 1: Retention-basin information extraction

Unstructured reports and papers often mention flood-retention basins, polders, and reservoirs without a database behind them. This exercise uses **DLLMForge** package and **Azure OpenAI model** to turn a PDF into structured JSON that matches a user-defined schema.

You will:

1. Inspect a hand-written Pydantic schema (identity + engineering + evidence).
2. Convert a PDF to text with `[PAGE N]` markers.
3. Extract records with `InfoExtractor`.
4. Compare the JSON to approximate values in `README.md` for Eupen and La Gileppe.

Prerequisites: course `uv sync` / hydrology-env verify is green, and Azure OpenAI variables are in the repo-root `.env`. Never paste API keys into this notebook.

## Step 1: Imports and Azure GPT


In [1]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from dllmforge.IE_agent_document_processor import DocumentProcessor, ProcessedDocument
from dllmforge.IE_agent_extractor import InfoExtractor
from dllmforge.langchain_api import LangchainAPI

EXERCISE_DIR = Path.cwd()
if not (EXERCISE_DIR / "schema.py").exists():
    EXERCISE_DIR = Path.cwd() / "exercises" / "01_information_extraction"
if str(EXERCISE_DIR) not in sys.path:
    sys.path.insert(0, str(EXERCISE_DIR))

from schema import WaterRetentionBasinDocument

print(f"Exercise dir: {EXERCISE_DIR}")
print("Imports OK")

Exercise dir: c:\Users\deng_jg\work\16centralized_agents\hydrology_user_day_training\LLMs-Agentic-Workflows-in-Hydrology-Training\exercises\01_information_extraction
Imports OK


c:\Users\deng_jg\work\16centralized_agents\hydrology_user_day_training\LLMs-Agentic-Workflows-in-Hydrology-Training\.venv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
c:\Users\deng_jg\work\16centralized_agents\hydrology_user_day_training\LLMs-Agentic-Workflows-in-Hydrology-Training\.venv\Lib\site-packages\dllmforge\IE_agent_config.py:36: UserWarning: Field name "schema" in "IEAgentConfig" shadows an attribute in parent "BaseModel"
  class IEAgentConfig(BaseModel):


In [2]:
load_dotenv()
if not os.getenv("AZURE_OPENAI_ENDPOINT") and os.getenv("AZURE_OPENAI_API_BASE"):
    os.environ["AZURE_OPENAI_ENDPOINT"] = os.environ["AZURE_OPENAI_API_BASE"]

llm_api = LangchainAPI(model_provider="azure-openai", temperature=0.1)
print("Azure OpenAI ready (", os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"), ")")

Azure OpenAI ready ( gpt-5.2 )


## Step 2: Schema

The schema is the contract: if a field is missing, the model will not return it. Let's print out the schema to inspect what information we are going to extract.

In [3]:
print("\nWaterRetentionBasin fields:")
from schema import WaterRetentionBasin

for name, info in WaterRetentionBasin.model_fields.items():
    print(f"  - {name}: {info.annotation}")


WaterRetentionBasin fields:
  - name: typing.Optional[str]
  - alternative_names: typing.Optional[typing.List[str]]
  - country: typing.Optional[str]
  - location: typing.Optional[str]
  - river_name: typing.Optional[str]
  - river_basin: typing.Optional[str]
  - catchment: typing.Optional[str]
  - flood_water_storage_volume_m3: typing.Optional[float]
  - storage_volume_total_m3: typing.Optional[float]
  - basin_control_type: typing.Optional[str]
  - dam_height_m: typing.Optional[float]
  - sill_level_m: typing.Optional[float]
  - operating_rules: typing.Optional[str]
  - activation_discharge_m3_s: typing.Optional[float]
  - activation_water_level_m: typing.Optional[float]
  - inlet_cross_section_m2: typing.Optional[float]
  - inlet_capacity_m3_s: typing.Optional[float]
  - source_page: typing.Optional[int]
  - source_quote: typing.Optional[str]


## Step 3: PDF to text with page markers

DLLMForge text extraction concatenates pages with **no page numbers**. The helper below inserts `[PAGE N]` so `source_page` can be grounded. `source_quote` is still filled by the LLM (the processor cannot pick the sentence).

In [4]:
import pymupdf as fitz

pdf_path = EXERCISE_DIR / "data" / "pdfs" / "nhess-15-365-2015.pdf"
output_dir = EXERCISE_DIR / "output"
output_dir.mkdir(parents=True, exist_ok=True)


def pdf_to_text_with_page_markers(path: Path) -> ProcessedDocument:
    path = Path(path)
    doc = fitz.open(str(path))
    parts = []
    for i, page in enumerate(doc, start=1):
        parts.append(f"\n[PAGE {i}]\n{page.get_text()}")
    text = "".join(parts)
    page_count = len(doc)
    doc.close()
    return ProcessedDocument(
        content=text,
        content_type="text",
        metadata={
            "source_file": str(path),
            "file_type": path.suffix.lower(),
            "text_length": len(text),
            "page_count": page_count,
        },
    )


processed = pdf_to_text_with_page_markers(pdf_path)
print(f"File: {pdf_path.name}")
print(f"Pages: {processed.metadata['page_count']}, characters: {processed.metadata['text_length']}")
print("--- first 800 characters ---")
print(processed.content[:800])

File: nhess-15-365-2015.pdf
Pages: 15, characters: 61872
--- first 800 characters ---

[PAGE 1]
Nat. Hazards Earth Syst. Sci., 15, 365–379, 2015
www.nat-hazards-earth-syst-sci.net/15/365/2015/
doi:10.5194/nhess-15-365-2015
© Author(s) 2015. CC Attribution 3.0 License.
Assessing the operation rules of a reservoir system based on a
detailed modelling chain
M. Bruwier, S. Erpicum, M. Pirotton, P. Archambeau, and B. J. Dewals
University of Liege (ULG), Hydraulics in Environmental and Civil Engineering (HECE), Liege, Belgium
Correspondence to: M. Bruwier (mbruwier@ulg.ac.be)
Received: 3 July 2014 – Published in Nat. Hazards Earth Syst. Sci. Discuss.: 9 September 2014
Revised: 4 February 2015 – Accepted: 6 February 2015 – Published: 2 March 2015
Abstract. According to available climate change scenarios
for Belgium, drier summers and wetter winters are expected.
In this study, we 


## Step 4: Build the extractor

Define system prompt for the extraction task.

In [5]:
system_prompt = """Extract every water retention basin, polder, or multi-purpose reservoir
mentioned in the document into the provided schema.

Rules:
- One list entry per distinct site. Do not merge two dams or basins (e.g. keep Eupen and La Gileppe separate).
- Only extract values that are explicitly stated. Use null if a field is missing or uncertain.
- Convert all volumes to cubic metres (m³). 1 hm³ = 1e6 m³, 1 Mm³ = 1e6 m³.
- river_name is the local river; river_basin is the main basin (e.g. Meuse).
- Set source_page from the [PAGE N] marker in the text only. Do not guess and do not use journal page numbers.
- Copy source_quote verbatim from the document (one sentence or a short paragraph). Do not paraphrase.

Return JSON only, matching the schema keys.
"""

doc_processor = DocumentProcessor(
    input_dir=str(EXERCISE_DIR / "data" / "pdfs"),
    file_pattern="*.pdf",
    output_type="text",
    output_dir=str(output_dir),
)

extractor = InfoExtractor(
    output_schema=WaterRetentionBasinDocument,
    llm_api=llm_api,
    system_prompt=system_prompt,
    chunk_size=80000,
    chunk_overlap=10000,
    doc_processor=doc_processor,
    document_output_type="text",
)
print("Extractor ready")

Document loader initialized with support for: .pdf, .xlsx, .xls, .csv
Extractor ready


## Step 5: Extract from one PDF

This cell calls Azure GPT and may take a minute.

In [6]:
results = extractor.process_document(processed)
output_path = output_dir / f"{pdf_path.stem}_extracted.json"
extractor.save_results(results, output_path)

print(f"Saved {len(results)} result object(s) to {output_path}")
print(json.dumps(results, indent=2, ensure_ascii=False, default=str))

INFO:httpx:HTTP Request: POST https://openai-dllmforge.openai.azure.com/openai/deployments/gpt-5.2/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


Results saved to c:\Users\deng_jg\work\16centralized_agents\hydrology_user_day_training\LLMs-Agentic-Workflows-in-Hydrology-Training\exercises\01_information_extraction\output\nhess-15-365-2015_extracted.json
Saved 1 result object(s) to c:\Users\deng_jg\work\16centralized_agents\hydrology_user_day_training\LLMs-Agentic-Workflows-in-Hydrology-Training\exercises\01_information_extraction\output\nhess-15-365-2015_extracted.json
[
  {
    "source_title": "Assessing the operation rules of a reservoir system based on a detailed modelling chain",
    "basins": [
      {
        "name": "Eupen dam",
        "alternative_names": [
          "Eupen reservoir"
        ],
        "country": "Belgium",
        "location": "situated on the main course of the River Vesdre, 3 km upstream of the town Eupen",
        "river_name": "Vesdre",
        "river_basin": "Meuse",
        "catchment": "Vesdre catchment",
        "flood_water_storage_volume_m3": 3000000.0,
        "storage_volume_total_m3": 25000

## Step 6: Inspect metadata vs evidence, then check 'groundtruth'values

- `metadata` comes from the document processor (file name, type, length, chunk offsets).
- `source_page` / `source_quote` come from the LLM. Open that PDF page and confirm the quote is really there.

Groundtruth values below are **approximate** (Bruwier et al., 2015). Volumes may be wrong if the model forgot to convert hm³ / Mm³ to m³.

In [7]:
basins = []
for result in results:
    print("Processor metadata:", result.get("metadata"))
    basins.extend(result.get("basins") or [])

print(f"\nExtracted {len(basins)} basin(s)")
for basin in basins:
    print("-" * 60)
    print(f"name:         {basin.get('name')}")
    print(f"country:      {basin.get('country')}")
    print(f"river_name:   {basin.get('river_name')}")
    print(f"river_basin:  {basin.get('river_basin')}")
    print(f"control:      {basin.get('basin_control_type')}")
    print(f"total m3:     {basin.get('storage_volume_total_m3')}")
    print(f"flood m3:     {basin.get('flood_water_storage_volume_m3')}")
    print(f"source_page:  {basin.get('source_page')}")
    print(f"source_quote: {basin.get('source_quote')}")

GOLD_VALUES = [
    {"name": "Eupen dam", "country": "Belgium", "river_name": "Vesdre", "river_basin": "Meuse", "basin_control_type": "controlled"},
    {"name": "La Gileppe dam", "country": "Belgium", "river_name": "Vesdre", "river_basin": "Meuse", "basin_control_type": "controlled"},
]
print("\nGold-value check:")
print(f"{'expected':<22} {'extracted name':<28} {'country':<12} {'river':<12} {'type':<12}")
for gold in GOLD_VALUES:
    needle = gold["name"].split()[0].lower()
    match = next((b for b in basins if needle in (b.get("name") or "").lower()), {})
    print(
        f"{gold['name']:<22} "
        f"{str(match.get('name') or 'NOT FOUND'):<28} "
        f"{str(match.get('country') or ''):<12} "
        f"{str(match.get('river_name') or ''):<12} "
        f"{str(match.get('basin_control_type') or ''):<12}"
    )

Processor metadata: {'source_file': 'c:\\Users\\deng_jg\\work\\16centralized_agents\\hydrology_user_day_training\\LLMs-Agentic-Workflows-in-Hydrology-Training\\exercises\\01_information_extraction\\data\\pdfs\\nhess-15-365-2015.pdf', 'file_type': '.pdf', 'text_length': 61872, 'page_count': 15, 'chunk_start': 0, 'chunk_end': 80000}

Extracted 2 basin(s)
------------------------------------------------------------
name:         Eupen dam
country:      Belgium
river_name:   Vesdre
river_basin:  Meuse
control:      controlled
total m3:     25000000.0
flood m3:     3000000.0
source_page:  4
source_quote: Two 50 m high dams are located in the upper part of the catchment: Eupen dam and La Gileppe dam. The former is situated on the main course of the River Vesdre, 3 km upstream of the town Eupen, while the latter is on the left-bank tributary La Gileppe. Both reservoirs have approximately the same storage capacity equal to 25 hm3.
------------------------------------------------------------
na

## Step 7 (optional): batch `process_all()`

If you add a second PDF under `data/pdfs/`, this uses DLLMForge's directory loop (plain text, **no** `[PAGE N]` markers). Skip it for the required path.

In [8]:
RUN_PROCESS_ALL = False

if RUN_PROCESS_ALL:
    extractor.process_all()
    print(f"Combined JSON is in {output_dir}")
else:
    print("Skipped process_all(). Set RUN_PROCESS_ALL = True after adding another PDF.")

Skipped process_all(). Set RUN_PROCESS_ALL = True after adding another PDF.


## Advanced: vision extraction (optional)

Completing the exercise does **not** require this section.

Use vision when:

- the PDF is scanned (little or no extractable text),
- numbers live only in a figure or table,
- text extraction returned almost nothing.

That is the same idea as the flood-measures image fallback, without running the full pipeline.

DLLMForge turns **each page** into a high-zoom JPEG. A full paper is one Azure multimodal call per page. Here we send **two pages** only.

Provenance difference vs the text path: image mode already puts `page_number` on processor `metadata`. `source_quote` is still an LLM field (the model is looking at pixels, not `[PAGE N]` markers).

Set `RUN_VISION = True` only if your Azure deployment accepts image inputs. If the call fails, read the skip message rather than retrying the whole PDF.

In [9]:
RUN_VISION = True
VISION_PAGES = (1, 2)  # 1-based PDF page numbers

if not RUN_VISION:
    print("Skipped vision extraction. Set RUN_VISION = True to try 1–2 pages.")
else:
    image_processor = DocumentProcessor(
        input_dir=str(EXERCISE_DIR / "data" / "pdfs"),
        file_pattern="*.pdf",
        output_type="image",
        output_dir=str(output_dir),
    )
    try:
        page_images = image_processor.process_to_image(pdf_path)
        selected = [
            page for page in page_images
            if page.metadata.get("page_number") in VISION_PAGES
        ]
        print(f"Sending {len(selected)} page image(s) to Azure GPT: {VISION_PAGES}")
        vision_extractor = InfoExtractor(
            output_schema=WaterRetentionBasinDocument,
            llm_api=llm_api,
            system_prompt=system_prompt,
            chunk_size=80000,
            chunk_overlap=10000,
            doc_processor=image_processor,
            document_output_type="image",
        )
        vision_results = vision_extractor.process_document(selected)
        vision_path = output_dir / f"{pdf_path.stem}_vision_extracted.json"
        vision_extractor.save_results(vision_results, vision_path)
        print(json.dumps(vision_results, indent=2, ensure_ascii=False, default=str))
        print("\nCompare with the text extraction: what did vision catch or miss?")
    except Exception as exc:
        print(
            "Vision extraction skipped: this Azure deployment may not accept image inputs, "
            f"or the page conversion failed.\n{type(exc).__name__}: {exc}"
        )

Document loader initialized with support for: .pdf, .xlsx, .xls, .csv
Sending 2 page image(s) to Azure GPT: (1, 2)


INFO:httpx:HTTP Request: POST https://openai-dllmforge.openai.azure.com/openai/deployments/gpt-5.2/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openai-dllmforge.openai.azure.com/openai/deployments/gpt-5.2/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


Results saved to c:\Users\deng_jg\work\16centralized_agents\hydrology_user_day_training\LLMs-Agentic-Workflows-in-Hydrology-Training\exercises\01_information_extraction\output\nhess-15-365-2015_vision_extracted.json
[
  {
    "source_title": "Assessing the operation rules of a reservoir system based on a detailed modelling chain",
    "basins": [
      {
        "name": null,
        "alternative_names": null,
        "country": "Belgium",
        "location": null,
        "river_name": null,
        "river_basin": "Meuse",
        "catchment": "Vesdre catchment",
        "flood_water_storage_volume_m3": null,
        "storage_volume_total_m3": null,
        "basin_control_type": null,
        "dam_height_m": null,
        "sill_level_m": null,
        "operating_rules": null,
        "activation_discharge_m3_s": null,
        "activation_water_level_m": null,
        "inlet_cross_section_m2": null,
        "inlet_capacity_m3_s": null,
        "source_page": null,
        "source_quote